# Desafio 1 — Prata: Frentes Parlamentares

## Objetivo
Limpar, padronizar e tipar os dados brutos da camada bronze, preparando para análises.

## Transformações aplicadas
- Extração de campos aninhados do JSON bruto via Python (json.loads)
- Join entre frentes e membros para identificar coordenadores
- Padronização de nomes de colunas (snake_case)
- Conversão de datas (string → DATE)
- Remoção de duplicatas via GROUP BY + MAX

## Decisões técnicas
- Coordenadores extraídos da tabela de membros (não do endpoint de detalhes)
- 102 frentes sem coordenador identificado (mantidas com NULL)


In [0]:
# ============================================================
# CAMADA PRATA - FRENTES (limpeza e padronizacao)
# ============================================================

# Ler dados brutos da bronze
df_bronze_frentes = spark.sql("SELECT * FROM bronze.frentes")

# Registrar como view temporaria para usar SQL
df_bronze_frentes.createOrReplaceTempView("bronze_frentes_view")

In [0]:
# ============================================================
# CAMADA PRATA - FRENTES (limpeza e padronizacao)
# ============================================================

# Ler dados brutos da bronze
df_bronze_frentes = spark.sql("SELECT * FROM bronze.frentes")

# Registrar como view temporaria para usar SQL
df_bronze_frentes.createOrReplaceTempView("bronze_frentes_view")


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS prata;
SHOW SCHEMAS;

# Desafio 2 — Prata:  Criar Tabela Prata

In [0]:
%sql
-- Criar tabela prata de membros com tipagem corrigida
CREATE OR REPLACE TABLE prata.frentes_membros
USING DELTA
COMMENT 'Camada Prata - Membros das frentes limpos e padronizados'
AS
SELECT
    -- Chave composta
    id_frente,
    id_deputado,
    
    -- Dados do deputado
    nome,
    sigla_partido,
    sigla_uf,
    
    -- Cargo na frente
    cargo,
    cod_cargo,
    
    -- Datas convertidas de string para date
    CASE 
        WHEN data_inicio IS NOT NULL AND data_inicio != '' 
        THEN CAST(data_inicio AS DATE) 
        ELSE NULL 
    END AS data_inicio,
    
    CASE 
        WHEN data_fim IS NOT NULL AND data_fim != '' 
        THEN CAST(data_fim AS DATE) 
        ELSE NULL 
    END AS data_fim,
    
    -- Metadados de auditoria
    data_coleta,
    origem
    
FROM bronze.frentes_membros;

In [0]:
%sql
SELECT 
    COUNT(*) AS total_registros,
    COUNT(DISTINCT id_frente) AS frentes,
    COUNT(DISTINCT id_deputado) AS deputados,
    COUNT(DISTINCT sigla_partido) AS partidos,
    COUNT(DISTINCT sigla_uf) AS ufs
FROM prata.frentes_membros;

In [0]:
%sql
-- Verificar se ha deputados sem partido ou UF
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN sigla_partido IS NULL THEN 1 ELSE 0 END) AS sem_partido,
    SUM(CASE WHEN sigla_uf IS NULL THEN 1 ELSE 0 END) AS sem_uf
FROM prata.frentes_membros;

In [0]:
%sql
SELECT 
    id_frente, 
    id_deputado, 
    cargo, 
    COUNT(*) AS ocorrencias
FROM prata.frentes_membros
GROUP BY id_frente, id_deputado, cargo
HAVING COUNT(*) > 1
LIMIT 10;

In [0]:
import json

# Carregar dados da bronze
df_bronze = spark.sql("SELECT * FROM bronze.frentes")
linhas = df_bronze.collect()

# Extrair campos do JSON bruto via Python
dados_prata = []
for row in linhas:
    try:
        bruto = json.loads(row['dado_bruto_json'])
    except:
        bruto = {}
    
    coordenador = bruto.get('coordenador') or {}
    
    dados_prata.append((
        row['id_frente'],
        row['titulo'],
        row['id_legislatura'],
        coordenador.get('id'),
        coordenador.get('nome'),
        coordenador.get('siglaPartido'),
        coordenador.get('siglaUf'),
        bruto.get('telefone'),
        bruto.get('email'),
        bruto.get('situacao'),
        bruto.get('keywords'),
        bruto.get('urlWebsite'),
        bruto.get('urlDocumento'),
        row['data_coleta'],
        row['origem']
    ))

print(f"Registros processados: {len(dados_prata)}")
print(f"Exemplo coordenador: ID={dados_prata[0][3]}, Nome={dados_prata[0][4]}, Partido={dados_prata[0][5]}")

In [0]:
# Criar prata.frentes a partir da bronze + join com membros para pegar coordenador
import json

# 1. Carregar frentes da bronze
df_bronze = spark.sql("SELECT * FROM bronze.frentes")
linhas = df_bronze.collect()

# 2. Carregar membros da prata para achar coordenadores
df_membros = spark.sql("""
    SELECT DISTINCT id_frente, id_deputado, nome, sigla_partido, sigla_uf
    FROM prata.frentes_membros
    WHERE cargo = 'Coordenador'
""")
coord_por_frente = {}
for row in df_membros.collect():
    coord_por_frente[row['id_frente']] = {
        'id': row['id_deputado'],
        'nome': row['nome'],
        'partido': row['sigla_partido'],
        'uf': row['sigla_uf']
    }

# 3. Montar dados da prata
dados_prata = []
for row in linhas:
    coord = coord_por_frente.get(row['id_frente'], {})
    dados_prata.append((
        row['id_frente'],
        row['titulo'],
        row['id_legislatura'],
        coord.get('id'),
        coord.get('nome'),
        coord.get('partido'),
        coord.get('uf'),
        row['data_coleta'],
        row['origem']
    ))

print(f"Registros processados: {len(dados_prata)}")
print(f"Exemplo: ID={dados_prata[0][0]}, Coordenador={dados_prata[0][4]}, Partido={dados_prata[0][5]}, UF={dados_prata[0][6]}")

In [0]:
# Verificar quantas frentes tem coordenador e quantas nao
com_coord = sum(1 for d in dados_prata if d[4] is not None)
sem_coord = sum(1 for d in dados_prata if d[4] is None)

print(f"Frentes COM coordenador encontrado: {com_coord}")
print(f"Frentes SEM coordenador encontrado: {sem_coord}")

# Ver um exemplo de frente que tem coordenador
for d in dados_prata:
    if d[4] is not None:
        print(f"\nExemplo com coordenador: ID={d[0]}, Coordenador={d[4]}, Partido={d[5]}, UF={d[6]}")
        break

In [0]:
from pyspark.sql.types import StructType, StructField, LongType, StringType, IntegerType

schema = StructType([
    StructField("id_frente", LongType(), True),
    StructField("titulo", StringType(), True),
    StructField("id_legislatura", IntegerType(), True),
    StructField("id_coordenador", LongType(), True),
    StructField("nome_coordenador", StringType(), True),
    StructField("partido_coordenador", StringType(), True),
    StructField("uf_coordenador", StringType(), True),
    StructField("data_coleta", StringType(), True),
    StructField("origem", StringType(), True)
])

df_prata = spark.createDataFrame(dados_prata, schema=schema)

df_prata.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("prata.frentes")

print(f"Tabela prata.frentes salva com {df_prata.count()} registros.")

In [0]:
%sql
CREATE OR REPLACE TABLE prata.frentes_membros
USING DELTA
COMMENT 'Camada Prata - Membros das frentes limpos, padronizados e sem duplicatas'
AS
SELECT
    id_frente,
    id_deputado,
    MAX(nome) AS nome,
    MAX(sigla_partido) AS sigla_partido,
    MAX(sigla_uf) AS sigla_uf,
    cargo,
    MAX(cod_cargo) AS cod_cargo,
    MAX(data_inicio) AS data_inicio,
    MAX(data_fim) AS data_fim,
    MAX(data_coleta) AS data_coleta,
    MAX(origem) AS origem
FROM prata.frentes_membros
GROUP BY id_frente, id_deputado, cargo;

In [0]:
%sql
SELECT 
    *
FROM prata.frentes_membros
limit 10